# Membrane-Aware Pipeline Analysis

Colab-ready notebook for M1 target audit, M2 causal probe, and M3 usefulness probe outputs.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid')
except Exception:
    sns = None

RUN_ROOT = Path('runs/dvs128gesture')  # edit in Colab/Drive
AUDIT_DIR = RUN_ROOT / 'membrane_target_audit_T8'
PROBE_DIR = RUN_ROOT / 'membrane_probe_T8'
USEFULNESS_DIR = RUN_ROOT / 'membrane_usefulness_T8'

def read_csv(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() and path.stat().st_size else pd.DataFrame()

def read_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() and path.stat().st_size else {}


## M1 Target Audit


In [ ]:
audit_summary = read_csv(AUDIT_DIR / 'target_audit_summary.csv')
audit_stage = read_csv(AUDIT_DIR / 'target_audit_by_stage.csv')
audit_layer = read_csv(AUDIT_DIR / 'target_audit_by_layer.csv')
state_stats = read_csv(AUDIT_DIR / 'membrane_state_stats.csv')
print('audit_summary', audit_summary.shape)
print('audit_stage', audit_stage.shape)
print('audit_layer', audit_layer.shape)
display(audit_summary.head(20))

if not audit_layer.empty:
    pass_cols = [c for c in audit_layer.columns if c.startswith('passes_')]
    counts = audit_layer[pass_cols].apply(lambda s: s.astype(str).str.lower().isin(['true','1']).sum())
    counts.plot(kind='bar', figsize=(10,4), title='M1 criteria pass counts by layer')
    plt.ylabel('layer-target configs passed')
    plt.tight_layout(); plt.show()

    cols = [c for c in ['split','target','stage','layer','relative_gain_copy_vs_zero_raw_error_mean','autocorr_lag1','temporal_selectivity_shuffle_copy_raw_error','temporal_selectivity_reverse_copy_raw_error','near_zero_fraction','passes_m1'] if c in audit_layer]
    display(audit_layer.sort_values([c for c in ['split','passes_m1','relative_gain_copy_vs_zero_raw_error_mean'] if c in audit_layer], ascending=[True, False, False])[cols].head(50))


In [ ]:
if not state_stats.empty:
    cols = [c for c in ['stage','layer','target','abs_mean','std','near_zero_fraction','above_threshold_fraction','near_threshold_fraction','temporal_autocorr_lag1'] if c in state_stats]
    display(state_stats[cols].head(50))
    if {'target','abs_mean'}.issubset(state_stats.columns):
        state_stats.groupby('target')['abs_mean'].median().sort_values().plot(kind='barh', figsize=(8,4), title='Median abs_mean by target')
        plt.tight_layout(); plt.show()


## M2 Causal Learned Probe


In [ ]:
decision = read_csv(PROBE_DIR / 'decision_metrics.csv')
pred_err = read_csv(PROBE_DIR / 'prediction_error.csv')
curve = read_csv(PROBE_DIR / 'probe_learning_curve.csv')
summary = read_json(PROBE_DIR / 'probe_summary.json')
print('decision', decision.shape, 'prediction_error', pred_err.shape, 'curve', curve.shape)
if summary:
    display({k: summary.get(k) for k in ['eval_splits','eval_modes','predictor_types','histories','amplitude_loss_weights','decision_criteria']})

if not decision.empty:
    pass_cols = [c for c in decision.columns if c.startswith('passes_')]
    counts = decision[pass_cols].apply(lambda s: s.astype(str).str.lower().isin(['true','1']).sum())
    counts.plot(kind='bar', figsize=(10,4), title='M2 criteria pass counts')
    plt.ylabel('configs passed')
    plt.tight_layout(); plt.show()

    test = decision[decision.get('eval_split','') == 'test'].copy() if 'eval_split' in decision else decision.copy()
    rank_cols = [c for c in ['target','stage','layer','history','predictor_type','amplitude_loss_weight','normal_normalized_error','normal_raw_error_mean','normal_prediction_abs_ratio','relative_gain_vs_zero_raw_error_mean','relative_gain_vs_copy_previous_normalized_error','temporal_selectivity_shuffle_normalized_error','temporal_selectivity_reverse_normalized_error','passes_strict_probe'] if c in test]
    display(test.sort_values('normal_normalized_error', na_position='last')[rank_cols].head(50))


In [ ]:
if not decision.empty and 'eval_split' in decision:
    test = decision[decision['eval_split'] == 'test'].replace([float('inf'), -float('inf')], pd.NA).dropna(subset=['normal_normalized_error'])
    if not test.empty:
        fig, axes = plt.subplots(1, 3, figsize=(18,5))
        for ax, metric, title in [(axes[0],'normal_normalized_error','normal error'), (axes[1],'relative_gain_vs_copy_previous_normalized_error','gain vs copy'), (axes[2],'relative_gain_vs_zero_raw_error_mean','gain vs zero raw')]:
            if metric not in test: ax.axis('off'); continue
            labels = test['target'].astype(str)+' '+test['stage'].astype(str)+' h'+test['history'].astype(str)+' '+test['predictor_type'].astype(str)
            ax.scatter(test['amplitude_loss_weight'], test[metric])
            ax.set_xscale('log'); ax.set_title(title); ax.set_xlabel('amp weight'); ax.grid(True, alpha=.3)
            if 'gain' in metric: ax.axhline(0, color='black', ls='--', lw=1)
        plt.tight_layout(); plt.show()

if not curve.empty:
    plot = curve[curve.get('eval_split','') == 'test'].copy() if 'eval_split' in curve else curve.copy()
    if not plot.empty:
        plot['config'] = plot['target'].astype(str)+' '+plot['stage'].astype(str)+' h'+plot['history'].astype(str)+' '+plot['predictor_type'].astype(str)+' a='+plot['amplitude_loss_weight'].astype(str)
        best = plot.sort_values('normalized_error').groupby('config').head(1)['config'].head(8).tolist()
        fig, ax = plt.subplots(figsize=(10,5))
        for cfg, sub in plot[plot['config'].isin(best)].groupby('config'):
            ax.plot(sub['step'], sub['normalized_error'], marker='o', label=cfg)
        ax.set_title('M2 learning curves: normalized error'); ax.set_xlabel('step'); ax.legend(fontsize=8, bbox_to_anchor=(1.02,1)); plt.tight_layout(); plt.show()


## M3 Usefulness Probe


In [ ]:
usefulness = read_csv(USEFULNESS_DIR / 'usefulness_summary.csv')
samples = read_csv(USEFULNESS_DIR / 'usefulness_samples.csv')
print('usefulness', usefulness.shape, 'samples', samples.shape)
display(usefulness)
if not samples.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15,4))
    samples.plot.scatter(x='prediction_error', y='confidence', ax=axes[0], alpha=.4, title='error vs confidence')
    samples.plot.scatter(x='prediction_error', y='entropy', ax=axes[1], alpha=.4, title='error vs entropy')
    if 'logit_stability_l1' in samples:
        samples.plot.scatter(x='prediction_error', y='logit_stability_l1', ax=axes[2], alpha=.4, title='error vs logit stability')
    plt.tight_layout(); plt.show()
    samples.assign(error_bin=pd.qcut(samples['prediction_error'], 4, duplicates='drop')).groupby('error_bin')['correct'].mean().plot(kind='bar', figsize=(8,4), title='accuracy by prediction-error quartile')
    plt.ylabel('accuracy'); plt.tight_layout(); plt.show()
